# Pipeline GDELT - Traitement des données

Objectif :
- Collecter les données GDELT
- Nettoyer et transformer
- Enrichir les données
- Sauvegarder en format exploitable

In [5]:
#Cellule pour l'importation des biliotheques
import pandas as pd          # Importer pandas pour manipuler les données en tableaux (DataFrames)
import numpy as np           # Importer numpy pour les calculs numériques et gestion des valeurs manquantes
from tqdm import tqdm        # Importer tqdm pour afficher des barres de progression
import os                    # Importer os pour interagir avec le système de fichiers (créer dossiers, vérifier tailles)
from pathlib import Path


# Afficher toutes les colonnes sans limites
pd.set_option('display.max_columns', None)

In [6]:
#Cellule pour le chargement de la dataset brut
df_raw = pd.read_csv(r"C:\Users\YAOITCHA Rosine\Hackathon_Isheero_Team04\data\raw\donnees_benin.csv", low_memory=False) # Lire le fichier CSV brut et le stocker dans un DataFrame

print(f" Lignes : {df_raw.shape[0]}")     # Afficher le nombre de lignes du DataFrame
print(f" Colonnes : {df_raw.shape[1]}")   # Afficher le nombre de colonnes du DataFrame

print(f"\n Colonnes disponibles :")
print(df_raw.columns.tolist())              # Afficher la liste de toutes les colonnes disponibles

 Lignes : 23859
 Colonnes : 17

 Colonnes disponibles :
['GLOBALEVENTID', 'SQLDATE', 'MONTHYEAR', 'YEAR', 'Actor1CountryCode', 'Actor2CountryCode', 'EventRootCode', 'EventBaseCode', 'GoldsteinScale', 'NumMentions', 'NumSources', 'NumArticles', 'AvgTone', 'ActionGeo_CountryCode', 'ActionGeo_Lat', 'ActionGeo_Long', 'SOURCEURL']


In [7]:
#Cellule pour le diagnostic de la qualite de la dataset
print("=== VALEURS MANQUANTES ===")

missing = df_raw.isnull().sum()             # Compter le nombre de valeurs manquantes (NaN) par colonne

missing_pct = (missing / len(df_raw) * 100).round(2)   # Calculer le pourcentage de valeurs manquantes par colonne, arrondi à 2 décimales

rapport_qualite = pd.DataFrame({            
    'manquants': missing,                   # Colonne avec le nombre de valeurs manquantes
    'pourcentage': missing_pct              # Colonne avec le pourcentage de valeurs manquantes
}).sort_values('pourcentage', ascending=False)  # Créer un DataFrame récapitulatif trié du plus au moins de valeurs manquantes

print(rapport_qualite[rapport_qualite['manquants'] > 0])  # Afficher uniquement les colonnes qui ont au moins une valeur manquante

print("\n=== TYPES DE COLONNES ===")
print(df_raw.dtypes)        # Afficher le type de données de chaque colonne (int, float, object...)

print("\n=== APERÇU ===")
df_raw.head(3)              # Afficher les 3 premières lignes pour vérifier visuellement les données

=== VALEURS MANQUANTES ===
                   manquants  pourcentage
Actor2CountryCode      13814        57.90
Actor1CountryCode      11633        48.76

=== TYPES DE COLONNES ===
GLOBALEVENTID              int64
SQLDATE                    int64
MONTHYEAR                  int64
YEAR                       int64
Actor1CountryCode            str
Actor2CountryCode            str
EventRootCode              int64
EventBaseCode              int64
GoldsteinScale           float64
NumMentions                int64
NumSources                 int64
NumArticles                int64
AvgTone                  float64
ActionGeo_CountryCode        str
ActionGeo_Lat            float64
ActionGeo_Long           float64
SOURCEURL                    str
dtype: object

=== APERÇU ===


,GLOBALEVENTID,SQLDATE,MONTHYEAR,YEAR,Actor1CountryCode,Actor2CountryCode,EventRootCode,EventBaseCode,GoldsteinScale,NumMentions,NumSources,NumArticles,AvgTone,ActionGeo_CountryCode,ActionGeo_Lat,ActionGeo_Long,SOURCEURL
0,1249903377,20250702,202507,2025,NaN,NaN,1,10,0.0,2,1,2,2.431611,BN,9.5,2.25,https://www.thecable.ng/tinubus-reforms-fuelli...
1,1249986577,20250703,202507,2025,NGA,NaN,1,15,0.0,1,1,1,-11.357341,BN,9.5,2.25,https://leadership.ng/court-jails-2-beninese-o...
2,1249984164,20250703,202507,2025,NaN,NaN,3,36,4.0,10,1,10,6.201550,BN,9.5,2.25,https://www.afrik.com/le-benin-au-women-s-pavi...


In [8]:
#Cellule pour le nettoyage de la dataset
df = df_raw.copy()          # Créer une copie du DataFrame brut pour ne pas modifier l'original

colonnes_cles = [           # Définir la liste des colonnes utiles pour notre analyse
    'GLOBALEVENTID',        # Identifiant unique de chaque événement
    'SQLDATE',              # Date de l'événement au format YYYYMMDD
    'MONTHYEAR',            # Mois et année de l'événement (ex: 202501)
    'YEAR',                 # Année seule de l'événement
    'Actor1CountryCode',    # Code pays du premier acteur impliqué
    'Actor2CountryCode',    # Code pays du deuxième acteur impliqué
    'EventRootCode',        # Code racine du type d'événement (classification CAMEO)
    'EventBaseCode',        # Code détaillé du type d'événement
    'GoldsteinScale',       # Score de stabilité de l'événement (de -10 à +10)
    'NumMentions',          # Nombre de fois que l'événement est mentionné dans les médias
    'NumSources',           # Nombre de sources différentes ayant couvert l'événement
    'NumArticles',          # Nombre d'articles ayant couvert l'événement
    'AvgTone',              # Ton moyen des articles (négatif = mauvais, positif = bon)
    'ActionGeo_CountryCode',# Code pays où l'événement s'est produit (doit être 'BC' pour Bénin)
    'ActionGeo_Lat',        # Latitude géographique de l'événement (pour les cartes)
    'ActionGeo_Long',       # Longitude géographique de l'événement (pour les cartes)
    'SOURCEURL'             # URL de l'article source original
]

colonnes_presentes = [c for c in colonnes_cles if c in df.columns] # Filtrer pour ne garder que les colonnes qui existent réellement dans notre CSV
# (évite une erreur si une colonne attendue est absente)

df = df[colonnes_presentes]   # Réduire le DataFrame aux seules colonnes utiles

df['SQLDATE'] = pd.to_datetime(df['SQLDATE'], format='%Y%m%d', errors='coerce')    # Convertir la colonne SQLDATE en format datetime Python
# format='%Y%m%d' indique que la date est au format YYYYMMDD (ex: 20250115)
# errors='coerce' remplace les dates invalides par NaT au lieu de planter

df['GoldsteinScale'] = pd.to_numeric(df['GoldsteinScale'], errors='coerce')     # Convertir GoldsteinScale en nombre décimal
# errors='coerce' remplace les valeurs non-numériques par NaN

df['AvgTone'] = pd.to_numeric(df['AvgTone'], errors='coerce')       # Convertir AvgTone en nombre décimal pour pouvoir faire des calculs dessus

df['ActionGeo_Lat'] = pd.to_numeric(df['ActionGeo_Lat'], errors='coerce')    # Convertir la latitude en nombre décimal pour l'utiliser dans les cartes

df['ActionGeo_Long'] = pd.to_numeric(df['ActionGeo_Long'], errors='coerce')      # Convertir la longitude en nombre décimal pour l'utiliser dans les cartes

nb_avant = len(df)                          # Mémoriser le nombre de lignes avant suppression des doublons
df = df.drop_duplicates(subset=['GLOBALEVENTID'])    # Supprimer les lignes en double en se basant sur l'identifiant unique de l'événement
print(f"Doublons supprimés : {nb_avant - len(df)}")   # Afficher combien de doublons ont été supprimés

df = df.dropna(subset=['SQLDATE'])       # Supprimer les lignes dont la date est invalide ou manquante (inutilisables pour l'analyse temporelle)

print(f"\n Dataset propre : {df.shape[0]} lignes x {df.shape[1]} colonnes")     # Afficher la taille finale du DataFrame nettoyé

print(f" Période couverte : {df['SQLDATE'].min()} → {df['SQLDATE'].max()}")    # Afficher la plage de dates couverte par les données

Doublons supprimés : 0

 Dataset propre : 23859 lignes x 17 colonnes
 Période couverte : 2025-01-01 00:00:00 → 2025-12-31 00:00:00


In [9]:
#Cellule pour l'enrichissement de la dataset
df['mois'] = df['SQLDATE'].dt.month         # Extraire le numéro du mois (1 à 12) depuis la date pour les analyses mensuelles

df['trimestre'] = df['SQLDATE'].dt.quarter  # Extraire le trimestre (1 à 4) depuis la date pour les analyses trimestrielles

df['annee'] = df['SQLDATE'].dt.year         # Extraire l'année depuis la date

df['mois_annee'] = df['SQLDATE'].dt.to_period('M').astype(str)  # Créer une colonne "YYYY-MM" (ex: "2025-03") pour grouper les données par mois sur les graphiques

def categoriser_ton(tone):                  # Définir une fonction pour catégoriser le ton médiatique
    if pd.isna(tone):                       # Si la valeur est manquante (NaN)
        return 'inconnu'                    # Retourner 'inconnu'
    elif tone > 1:                          # Si le ton est supérieur à 1
        return 'positif'                    # L'article parle positivement du Bénin
    elif tone < -1:                         # Si le ton est inférieur à -1
        return 'negatif'                    # L'article parle négativement du Bénin
    else:                                   # Sinon (entre -1 et 1)
        return 'neutre'                     # Le ton est neutre

df['ton_categorie'] = df['AvgTone'].apply(categoriser_ton)  # Appliquer la fonction à chaque ligne de la colonne AvgTone pour créer la colonne ton_categorie

def categoriser_goldstein(score):           # Définir une fonction pour catégoriser le score Goldstein
    if pd.isna(score):                      # Si la valeur est manquante
        return 'inconnu'                    # Retourner 'inconnu'
    elif score > 3:                         # Si le score est supérieur à 3
        return 'cooperatif'                 # L'événement est de nature coopérative (accord, aide...)
    elif score < -3:                        # Si le score est inférieur à -3
        return 'conflictuel'                # L'événement est de nature conflictuelle (violence, crise...)
    else:                                   # Sinon (entre -3 et 3)
        return 'neutre'                     # L'événement est de nature neutre

df['type_event'] = df['GoldsteinScale'].apply(categoriser_goldstein)  # Appliquer la fonction à chaque ligne de GoldsteinScale pour créer la colonne type_event

print(" Colonnes enrichies ajoutées")     # Confirmer que l'enrichissement est terminé
df.head(3)                                  # Afficher les 3 premières lignes pour vérifier les nouvelles colonnes

 Colonnes enrichies ajoutées


,GLOBALEVENTID,SQLDATE,MONTHYEAR,YEAR,Actor1CountryCode,Actor2CountryCode,EventRootCode,EventBaseCode,GoldsteinScale,NumMentions,NumSources,NumArticles,AvgTone,ActionGeo_CountryCode,ActionGeo_Lat,ActionGeo_Long,SOURCEURL,mois,trimestre,annee,mois_annee,ton_categorie,type_event
0,1249903377,2025-07-02,202507,2025,NaN,NaN,1,10,0.0,2,1,2,2.431611,BN,9.5,2.25,https://www.thecable.ng/tinubus-reforms-fuelli...,7,3,2025,2025-07,positif,neutre
1,1249986577,2025-07-03,202507,2025,NGA,NaN,1,15,0.0,1,1,1,-11.357341,BN,9.5,2.25,https://leadership.ng/court-jails-2-beninese-o...,7,3,2025,2025-07,negatif,neutre
2,1249984164,2025-07-03,202507,2025,NaN,NaN,3,36,4.0,10,1,10,6.201550,BN,9.5,2.25,https://www.afrik.com/le-benin-au-women-s-pavi...,7,3,2025,2025-07,positif,cooperatif


In [ ]:
#Cellule pour la sauvegarde des fichiers des nouvelles dataseta
CURRENT_DIR = Path.cwd() # Chemin courant (où le kernel exécute)
for parent in [CURRENT_DIR] + list(CURRENT_DIR.parents): #Retrouver la le dossier racine comportant les dossiers data et notebooks
    if (parent / "notebooks").exists() and (parent / "data").exists():
        ROOT_DIR = parent
        break
    
print("ROOT_DIR =", ROOT_DIR)
DATA_PROCESSED = ROOT_DIR / "data" / "processed" #Definir la variable recuperant le chemin d'acces vers le dossier processed

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)  # Creer le chemin vers le dossier processed s'il n'esxiste pas encore

df.to_csv(DATA_PROCESSED/"benin_clean.csv", index=False)        # Sauvegarder le DataFrame nettoyé en CSV
# index=False pour ne pas écrire l'index pandas dans le fichier

df.to_csv(DATA_PROCESSED/"benin_enrichi.csv", index=False)      # Sauvegarder le DataFrame enrichi (avec colonnes calculées) en CSV

df.to_parquet(DATA_PROCESSED/"benin_enrichi.parquet", index=False)  # Sauvegarder en format Parquet — plus compact et plus rapide à lire que CSV

print(f" benin_clean.csv sauvegardé")
print(f" benin_enrichi.csv sauvegardé")
print(f" benin_enrichi.parquet sauvegardé")

print(f"\n Taille CSV : {os.path.getsize(DATA_PROCESSED/"benin_enrichi.csv") / 1024:.1f} KB")   # Afficher la taille du fichier CSV en kilooctets pour vérifier qu'il est raisonnable

CURRENT_DIR = Path.cwd()



ROOT_DIR = c:\Users\YAOITCHA Rosine\Hackathon_Isheero_Team04
 benin_clean.csv sauvegardé
 benin_enrichi.csv sauvegardé
 benin_enrichi.parquet sauvegardé

 Taille CSV : 5108.7 KB


In [15]:
#Rapport qualite final
print("=" * 50)                             # Afficher une ligne de séparation visuelle
print("RAPPORT QUALITÉ — PIPELINE v1")  # Titre du rapport
print("=" * 50)

print(f"Nombre d'événements    : {len(df):,}")         # Afficher le nombre total d'événements (le :, ajoute des séparateurs de milliers)

print(f"Période couverte       : {df['SQLDATE'].min().date()} → {df['SQLDATE'].max().date()}")    # Afficher la date du premier et dernier événement (.date() enlève l'heure)

print(f"Colonnes disponibles   : {df.shape[1]}")       # Afficher le nombre total de colonnes dans le DataFrame final

print(f"Valeurs nulles GoldsteinScale : {df['GoldsteinScale'].isna().sum()}")    # Afficher combien de valeurs manquantes restent dans GoldsteinScale

print(f"Valeurs nulles AvgTone        : {df['AvgTone'].isna().sum()}")     # Afficher combien de valeurs manquantes restent dans AvgTone

print(f"\nRépartition ton médiatique :")
print(df['ton_categorie'].value_counts())     # Afficher combien d'événements sont positifs, négatifs, neutres ou inconnus

print(f"\nRépartition type événement :")
print(df['type_event'].value_counts())      # Afficher combien d'événements sont coopératifs, conflictuels, neutres ou inconnus



RAPPORT QUALITÉ — PIPELINE v1
Nombre d'événements    : 23,859
Période couverte       : 2025-01-01 → 2025-12-31
Colonnes disponibles   : 23
Valeurs nulles GoldsteinScale : 0
Valeurs nulles AvgTone        : 0

Répartition ton médiatique :
ton_categorie
negatif    12619
positif     7258
neutre      3982
Name: count, dtype: int64

Répartition type événement :
type_event
neutre         11952
cooperatif      7007
conflictuel     4900
Name: count, dtype: int64
